# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library, referencing entities strictly by their Croissant `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets (tables), their fields (columns), and their Croissant `@id`s.

First, inspect which record sets are available, and then check their fields and sample data.

**Note:** All references use the Croissant `@id`.

In [ ]:
# List all record set IDs
print("Record sets and their @id values:")
record_sets = [r['@id'] for r in metadata.to_json().get('recordSet', [])]
if not record_sets:
    # Try the alternate key used by mlcroissant to match the Croissant 1.0 schema
    record_sets = [r['@id'] for r in metadata.to_json().get('recordSets', [])]

if record_sets:
    for i, rid in enumerate(record_sets):
        print(f"  {i+1}. {rid}")
else:
    print("No record sets found in Croissant schema.")

In [ ]:
# If there are record sets, list their fields' @id and name
example_record_set_id = None
fields_dict = {}
if record_sets:
    # Use the first record set for overview
    example_record_set_id = record_sets[0]
    print(f"\nFields for record set '{example_record_set_id}':")
    record_set_obj = None
    for rs in metadata.to_json().get('recordSet', []):
        if rs['@id'] == example_record_set_id:
            record_set_obj = rs
            break
    if not record_set_obj:
        for rs in metadata.to_json().get('recordSets', []):
            if rs['@id'] == example_record_set_id:
                record_set_obj = rs
                break
    if record_set_obj:
        fields = record_set_obj.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for f in fields:
            field_id = f['@id'] if isinstance(f, dict) else f
            fields_dict[field_id] = None
            # Try to get more info
        print("  Field @id values:")
        for i, field_id in enumerate(fields_dict.keys()):
            print(f"    {i+1}. {field_id}")
    else:
        print("Record set definition not found in metadata.")
else:
    print("No record sets to examine fields.")

In [ ]:
# Print a few records (first record set and its data using @id)
if example_record_set_id is not None:
    print(f"\nSample records from record set '@id': {example_record_set_id}")
    for i, rec in enumerate(dataset.records(record_set=example_record_set_id)):
        print(rec)
        if i > 2:
            break
else:
    print("No example record set found.")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis.

All record sets are referenced by their `@id`. Each DataFrame's columns are derived from the dataset's field `@id`s.

In [ ]:
dataframes = {}
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for record set '@id': {record_set_id} with shape {df.shape}")
            print(f"  Columns: {df.columns.tolist()}")
        else:
            print(f"Record set '@id': {record_set_id} is empty.")
    except Exception as e:
        print(f"Failed to load record set '@id': {record_set_id} ({e})")

# For this notebook, pick the first available non-empty record set for demonstration
target_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        target_record_set_id = rid
        break

if target_record_set_id:
    print(f"\nFirst DataFrame head for record set '@id': {target_record_set_id}")
    display(dataframes[target_record_set_id].head())
else:
    print("No data loaded into DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes.

All field accesses and DataFrame columns use their Croissant `@id`.

In [ ]:
if target_record_set_id is not None and not dataframes[target_record_set_id].empty:
    df = dataframes[target_record_set_id]

    # Find a numeric column
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) or (df[col].dropna().astype(str).str.replace('.', '', 1).str.isnumeric().all())]
    numeric_field_id = numeric_candidates[0] if numeric_candidates else None

    if numeric_field_id:
        print(f"Numeric field identified for EDA: {numeric_field_id}")

        # Coerce column to numeric (in case it's not already)
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        # Example threshold: mean for demonstration
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records ({numeric_field_id} > {threshold:.2f}): {len(filtered_df)} records.")

        # Normalize the numeric field
        filtered_df[numeric_field_id + "_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

        # Try grouping by another (likely categorical) column
        group_candidates = [col for col in df.columns if col != numeric_field_id]
        group_field_id = None
        for col in group_candidates:
            if df[col].nunique() < 20:
                group_field_id = col
                break

        if group_field_id:
            print(f"\nGrouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable group-by field identified.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No data for EDA.")

## 5. Visualization
Visualize data distributions or relationships. All axes and references use columns' Croissant `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if target_record_set_id is not None and not dataframes[target_record_set_id].empty and 'numeric_field_id' in locals() and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[target_record_set_id][numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[target_record_set_id].dropna(subset=[numeric_field_id, group_field_id]))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
This notebook demonstrated how to load, inspect, and perform basic analysis on the FAIR^2 dataset's record sets using the `mlcroissant` library.

Key findings:
- Croissant `@id`s ensure all access is robust and reproducible.
- The dataset's fields and record sets can be explored programmatically.
- Simple analyses such as filtering, normalization, grouping, and visualization can be performed with a few lines of code.

You may extend this notebook for deeper domain insights, advanced analytics, or richer visualization, leveraging the full semantic structure exposed by the Croissant schema.
